In [1]:
import sys
import matplotlib.pyplot as plt
import numpy as np

# Diamo a Python il percorso ASSOLUTO ed esatto del tuo progetto
sys.path.insert(0, '/home/jessica/TempSimul')

from lib.Dataloaders.VirtualAgentDataset import VirtualAgentDataset
import matplotlib.pyplot as plt

# 1. Carica il dataset
DATA_PATH = "/home/jessica/storage/VA_Dataset_Tensor.npz"
data = VirtualAgentDataset(DATA_PATH, split='train', norm=None)

from lib.Dataloaders.VirtualAgentDataset import VirtualAgentDataset
import matplotlib.pyplot as plt

# 1. Carica il dataset (usa il percorso assoluto per evitare problemi di path)
DATA_PATH = "/home/jessica/storage/VA_Dataset_Tensor.npz"
data = VirtualAgentDataset(DATA_PATH, split='train', norm=None)

# 2. Estrai il PRIMO campione tramite il metodo __getitem__
# x_endo avrà forma (8, 300) -> (Canali, Tempo)
# y_target avrà forma (1, 75) -> (Canale_Target, Tempo_Futuro)
x_endo, x_exo, x_cond, y_target = data[2000]  # Cambia l'indice per vedere campioni diversi
print(f"x_endo shape: {x_endo.shape}, x_exo shape: {x_exo.shape}, x_cond shape: {x_cond.shape}, y_target shape: {y_target.shape}")

# 3. Creiamo una figura con due grafici affiancati
plt.figure(figsize=(14, 4))

# --- PLOT 1: Il Passato (Lookback) ---
plt.subplot(1, 2, 1)
# Plottiamo il canale 0 (il 'Self') lungo tutto il lookback
plt.plot(x_endo[0, :].numpy(), color='blue', label='Self Dist3D (Passato)')
plt.title("Input X: Finestra di Lookback (300 frame)")
plt.xlabel("Frame (Tempo)")
plt.ylabel("Distanza (Z-Score)")
plt.legend()
plt.grid(True, alpha=0.3)

# --- PLOT 2: Il Futuro (Horizon) ---
plt.subplot(1, 2, 2)
# Plottiamo l'unico canale del target lungo tutto l'horizon
plt.plot(y_target[0, :].numpy(), color='orange', label='Self Dist3D (Futuro)')
plt.title("Target Y: Finestra di Horizon (75 frame)")
plt.xlabel("Frame (Tempo)")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

[TRAIN] Caricamento tensori da /home/jessica/storage/VA_Dataset_Tensor.npz in corso...


FileNotFoundError: [Errno 2] No such file or directory: '/home/jessica/storage/VA_Dataset_Tensor.npz'

In [ ]:
def debug_visivo_completo(dataset, idx=0, split_name="Dataset"):
    # Estraiamo i 4 tensori dal Dataloader
    x_endo, x_exo, x_cond, y_target = dataset[idx]
    
    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(f"Dashboard Debug - {split_name} | Indice Campione: {idx}", fontsize=16, fontweight='bold')

    # --- PLOT 1: X_endo (Il Passato: Distanze) ---
    ax1 = plt.subplot(2, 2, 1)
    ax1.plot(x_endo[0, :].numpy(), label='Self (Canale 0)', color='blue', linewidth=2)
    ax1.plot(x_endo[1, :].numpy(), label='Other 1 (Canale 1)', color='lightblue', alpha=0.8)
    ax1.set_title("X_endo: Distanze 3D Passate (Lookback)")
    ax1.set_xlabel("Frame (Tempo passato)")
    ax1.set_ylabel("Distanza Scalata [0.0, 1.0]")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # --- PLOT 2: X_exo (Variabili Ambientali) ---
    # Nuovi indici fisici: 0:Prua_X, 1:Prua_Z, 2:Boat_Angle, 3:Metronomo, 4:TeamScore, 5-18:Props&Raycast
    ax2 = plt.subplot(2, 2, 2)
    ax2.plot(x_exo[2, :].numpy(), label='Angolo Barca (Canale 2)', color='green', linewidth=1.5)
    ax2.plot(x_exo[3, :].numpy(), label='Metronomo (Canale 3)', color='red', linestyle='dashed')
    ax2.plot(x_exo[4, :].numpy(), label='TeamScore (Canale 4)', color='purple', linewidth=2)
    ax2.set_title("X_exo: Variabili Esogene (19 Canali)")
    ax2.set_xlabel("Frame (Tempo passato)")
    ax2.set_ylabel("Valore Normalizzato")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # --- PLOT 3: Y_target (Il Futuro) ---
    ax3 = plt.subplot(2, 2, 3)
    ax3.plot(y_target[0, :].numpy(), label='Target Futuro (Self)', color='orange', linewidth=2)
    ax3.set_title("Y_target: Orizzonte Futuro (Horizon)")
    ax3.set_xlabel("Frame (Tempo futuro)")
    ax3.set_ylabel("Distanza Scalata [0.0, 1.0]")
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # --- PLOT 4: X_cond (Variabili Statiche Condizionali) ---
    ax4 = plt.subplot(2, 2, 4)
    cond_np = x_cond.numpy()
    
    # Coloriamo diversamente i primi 8 (RowSide) e i secondi 8 (SpawnPoint)
    bar_colors = ['#800080']*8 + ['#008080']*8 # Viola per i Lati, Verde acqua per gli Spawn
    
    ax4.bar(range(len(cond_np)), cond_np, color=bar_colors, alpha=0.7)
    ax4.set_title("X_cond: Variabili Statiche (16 Canali)")
    ax4.set_xlabel("Indici (0-7: RowSide, 8-15: SpawnPoint)")
    ax4.set_xticks(range(len(cond_np)))
    
    # Creiamo una legenda custom per i colori
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#800080', label='RowSide'),
                       Patch(facecolor='#008080', label='SpawnPoint')]
    ax4.legend(handles=legend_elements, loc='upper right')
    ax4.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Crea le istanze del dataloader con la normalizzazione attiva
DATA_PATH = "/home/jessica/storage/VA_Dataset_Tensor.npz"
train_dataset = VirtualAgentDataset(DATA_PATH, split='train', norm=None)
val_dataset = VirtualAgentDataset(DATA_PATH, split='val', norm=None)
test_dataset = VirtualAgentDataset(DATA_PATH, split='test', norm=None)

In [ ]:
# 1. Guarda il campione 1500 del TRAINING
debug_visivo_completo(train_dataset, idx=131723, split_name="TRAINING")

# 2. Guarda il campione 400 della VALIDATION
debug_visivo_completo(val_dataset, idx=2822, split_name="VALIDATION")

# 3. Guarda il primissimo campione del TEST
debug_visivo_completo(test_dataset, idx=2822, split_name="TEST")